In [ ]:
import numpy as np
import plotly.express as px
import pandas as pd

# 1. Tạo dữ liệu giả lập

data = np.array([
    [200,1,1],
    [250,2,1],
    [100,1,-1],
    [150,0,-1],
    [150,1,1],
    [170,0,-1],
    [180,2,1],
    [140,2,1],
    [210,0,-1],
    [260,0,-1]
])

weight = data[:,0]
ripeness = data[:,2]  # target (0-1)
color_code = data[:,1] # có 3 màu (0 1 2)

X = np.column_stack((weight, color_code))  # shape (300,2)
y = ripeness.reshape(-1, 1)  # shape (300,1)


# 2. Đưa vào DataFrame cho tiện
df = pd.DataFrame({
    "Weight": weight,
    "ColorCode": color_code,
    "Ripeness": ripeness
})

#3. Vẽ 3D scatter bằng Plotly (xoay được bằng chuột)
fig = px.scatter_3d(
    df, x="Weight", y="ColorCode", z="Ripeness",
    color="Ripeness", color_continuous_scale="viridis",
    size_max=8, opacity=0.8
)

fig.update_layout(
    scene=dict(
        xaxis_title="Weight (g)",
        yaxis_title="Color Code",
        zaxis_title="Ripeness (0–1)"
    ),
    title="3D Visualization of Fruit Data (Interactive)",
    width = 900,
    height = 700
)

fig.show(config={"displaylogo": False, "modeBarButtonsToAdd": ["fullscreen"]})

In [4]:
X = data[:, :2].astype(float)   # features: khối lượng, màu sắc
y = data[:, 2]                  # nhãn -1 hoặc 1
num_samples, n_features = X.shape

# =======================
# 2. Chuẩn hóa dữ liệu (z-score)
X = (X - X.mean(axis=0)) / X.std(axis=0)

# =======================
# 3. Hàm sigmoid & loss
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_loss(y, z):
    return np.log(1 + np.exp(-y * z)).item()  # scalar

# =======================
# 4. Khởi tạo tham số
W = np.random.randn(n_features, 1) * 0.01
b = 0.0

# =======================
# 5. SGD Training
learning_rate = 0.1
epochs = 60

for epoch in range(epochs):
    loss_epoch = 0.0
    indices = np.random.permutation(num_samples)
    for i in indices:
        x_i = X[i].reshape(-1,1)   # (2,1)
        y_i = y[i]

        # logit
        z = float(W.T @ x_i + b)

        # loss
        loss_epoch += logistic_loss(y_i, z)

        # gradient
        grad_w = -(y_i * x_i) / (1 + np.exp(y_i * z))
        grad_b = -(y_i) / (1 + np.exp(y_i * z))

        # update
        W -= learning_rate * grad_w
        b -= learning_rate * grad_b

    avg_loss = loss_epoch / num_samples
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Avg Loss = {avg_loss:.4f}")

# =======================
# 6. Hàm dự đoán
def predict(X_input):
    p = sigmoid(X_input @ W + b)
    return np.where(p >= 0.5, 1, -1).flatten()

# =======================
# 7. Đánh giá
y_pred = predict(X)
acc = np.mean(y_pred == y)

print("\nFinal Weights:", W.flatten(), "Bias:", b)
print("Predicted:", y_pred)
print("True     :", y)
print(f"Final Accuracy: {acc:.4f}")

Epoch 0: Avg Loss = 0.6470
Epoch 20: Avg Loss = 0.2015
Epoch 40: Avg Loss = 0.1570

Final Weights: [1.11893691 4.04822045] Bias: 0.6018362267719326
Predicted: [ 1  1 -1 -1  1 -1  1  1 -1 -1]
True     : [ 1  1 -1 -1  1 -1  1  1 -1 -1]
Final Accuracy: 1.0000


# CROSS ENTROPY

In [4]:
X = data[:, :2].astype(float).copy()   # features: khối lượng, màu sắc
y = data[:, 2].copy()                  # nhãn -1 hoặc 1
y[y==-1] = 0
num_samples, n_features = X.shape

# =======================
# 2. Chuẩn hóa dữ liệu (z-score)
X = (X - X.mean(axis=0)) / X.std(axis=0)

# =======================
# 3. Hàm sigmoid & loss
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def BCE_loss(y, p):
    return - (y * np.log(p+10-6) + (1 - y) * np.log(1 - p +10-6)).item()  

# =======================
# 4. Khởi tạo tham số
W = np.random.randn(n_features, 1) * 0.01
b = 0.0

# =======================
# 5. SGD Training
learning_rate = 0.1
epochs = 60

for epoch in range(epochs):
    loss_epoch = 0.0
    indices = np.random.permutation(num_samples)
    for i in indices:
        x_i = X[i].reshape(-1,1)   # (2,1)
        y_i = y[i]

        # probability
        p = sigmoid(W.T @ x_i + b)

        # loss
        loss_epoch += BCE_loss(y_i, p)

        # gradient
        grad_w = (p-y_i) * x_i
        grad_b = (p-y_i)

        # update
        W -= learning_rate * grad_w
        b -= learning_rate * grad_b

    avg_loss = loss_epoch / num_samples
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Avg Loss = {avg_loss:.4f}")

# =======================
# 6. Hàm dự đoán
def predict(X_input):
    p = sigmoid(X_input @ W + b)
    return np.where(p >= 0.5, 1, 0).flatten()

# =======================
# 7. Đánh giá
y_pred = predict(X)
acc = np.mean(y_pred == y)

print("\nFinal Weights:", W.flatten(), "Bias:", b)
print("Predicted:", y_pred)
print("True     :", y)
print(f"Final Accuracy: {acc:.4f}")

Epoch 0: Avg Loss = -1.5094
Epoch 20: Avg Loss = -1.5756
Epoch 40: Avg Loss = -1.5826

Final Weights: [1.11856495 4.04782633] Bias: [[0.59895866]]
Predicted: [1 1 0 0 1 0 1 1 0 0]
True     : [1 1 0 0 1 0 1 1 0 0]
Final Accuracy: 1.0000


# Mean Square Error: Sigmoid function

In [9]:
X = data[:, :2].astype(float).copy()   # features: khối lượng, màu sắc
y = data[:, 2].copy()                  # nhãn -1 hoặc 1
y[y==-1] = 0
num_samples, n_features = X.shape

# =======================
# 2. Chuẩn hóa dữ liệu (z-score)
X = (X - X.mean(axis=0)) / X.std(axis=0)

# =======================
# 3. Hàm sigmoid & loss
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def BCE_loss(y, p):
    return - (y * np.log(p+10-6) + (1 - y) * np.log(1 - p +10-6)).item()  

# =======================
# 4. Khởi tạo tham số
W = np.random.randn(n_features, 1) * 0.01
b = 0.0

# =======================
# 5. SGD Training
learning_rate = 0.1
epochs = 60

for epoch in range(epochs):
    loss_epoch = 0.0
    indices = np.random.permutation(num_samples)
    for i in indices:
        x_i = X[i].reshape(-1,1)   # (2,1)
        y_i = y[i]

        # probability
        p = sigmoid(W.T @ x_i + b)

        # loss
        loss_epoch += BCE_loss(y_i, p)

        # gradient
        grad_w = -2*(y_i-p)*p*(1-p) * x_i
        grad_b = -2*(y_i-p)*p*(1-p)

        # update
        W -= learning_rate * grad_w
        b -= learning_rate * grad_b

    avg_loss = loss_epoch / num_samples
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Avg Loss = {avg_loss:.4f}")

# =======================
# 6. Hàm dự đoán
def predict(X_input):
    p = sigmoid(X_input @ W + b)
    return np.where(p >= 0.5, 1, 0).flatten()

# =======================
# 7. Đánh giá
y_pred = predict(X)
acc = np.mean(y_pred == y)

print("\nFinal Weights:", W.flatten(), "Bias:", b)
print("Predicted:", y_pred)
print("True     :", y)
print(f"Final Accuracy: {acc:.4f}")

Epoch 0: Avg Loss = -1.5068
Epoch 20: Avg Loss = -1.5608
Epoch 40: Avg Loss = -1.5697

Final Weights: [0.6606799  2.46687155] Bias: [[0.31425191]]
Predicted: [1 1 0 0 1 0 1 1 0 0]
True     : [1 1 0 0 1 0 1 1 0 0]
Final Accuracy: 1.0000
